In [17]:
import os
import numpy as np

# Image segmentation
from scipy import ndimage

# Image processing and feature extraction modules
import skimage.measure as skm
import skimage as ski

# Resize images with padding
from PIL import Image, ImageOps

In [18]:
folder_path = "../../images/"
images_names = sorted([f for f in os.listdir(folder_path) if f.endswith('.png')])

images = {}

for image in images_names:
    full_path = os.path.join(folder_path + image)
    try:
        name_key = os.path.basename(full_path).split('.')[0]
        images[name_key] = ski.io.imread(full_path)
    except Exception as e:
        print(e)

## `extract_shape_feature` function

In [22]:
def extract_shape_features(img):
    img_filled = ndimage.binary_fill_holes(img)
    label_img = skm.label(img_filled, connectivity=img_filled.ndim)
    props = skm.regionprops(label_img)
    if len(props) == 0:
        return None
    main_lesion = max(props, key=lambda x: x.area)
    features = {
        "Area": main_lesion.area,
        "Area Bounding Box": main_lesion.area_bbox,
        "Area Convex": main_lesion.area_convex,
        "Area Filled": main_lesion.area_filled,
        "Axis Major Length": main_lesion.axis_major_length,
        "Axis Minor Length": main_lesion.axis_minor_length,
        "Centroid Local": main_lesion.centroid_local,
        "Eccentricity": main_lesion.eccentricity,
        "Extent": main_lesion.extent,
        "Equivalent Diameter": main_lesion.equivalent_diameter,
        "Euler Number": main_lesion.euler_number,
        "Feret Diameter Max": main_lesion.feret_diameter_max,
        "Perimeter": main_lesion.perimeter,
        "Solidity": main_lesion.solidity,

        # Manually calculated features as regionprops module does not support them

        "Roundness": (4*main_lesion.area) / (np.pi * (main_lesion.axis_major_length**2)),
        "Circularity": (4*np.pi*main_lesion.area)/(main_lesion.perimeter**2),
        "Aspect Ratio": main_lesion.axis_major_length / main_lesion.axis_minor_length,
        "Irregularity Index": main_lesion.perimeter / (2 * np.sqrt(np.pi * main_lesion.area))
    }
    
    return features

test_extract = extract_shape_features(images['1008'])
for i in test_extract.items():
    print(f'{i[0]} : {i[1]}')


Area : 42718.0
Area Bounding Box : 59520.0
Area Convex : 48766.0
Area Filled : 42718.0
Axis Major Length : 249.8862945404044
Axis Minor Length : 221.72901262511724
Centroid Local : [121.02408821 123.1665106 ]
Eccentricity : 0.4611549489255556
Extent : 0.7177083333333333
Equivalent Diameter : 233.21716676093266
Euler Number : 1
Feret Diameter Max : 271.01660465735307
Perimeter : 1774.7018023401638
Solidity : 0.8759791658122462
Roundness : 0.8710361022495339
Circularity : 0.17043939922551377
Aspect Ratio : 1.1269896148542968
Irregularity Index : 2.4222279026832583


## `resize_with_padding` function

In [20]:
def resize_with_padding(img_path, target_size):
    img = Image.open(img_path).convert('RGB')
    result = ImageOps.pad(img, (target_size, target_size), method=Image.Resampling.LANCZOS, color='black', centering=(0.5, 0.5))
    return result

res = resize_with_padding("../../images/1472.png", 224)
res.show()
res = np.array(res)
res.shape

(224, 224, 3)